# Funtions used in this analysis

## Notes before running the scripts
- Before running codes, please set WORKSPACE_CDR to C2024Q3R9
- Depedencies: mpire, xlswriter, pyarrow, fastparquet


In [ ]:
!pip install xlsxwriter
!pip install mpire
!pip install pyarrow
!pip install fastparquet
!pip install papermill

In [ ]:
import sys
import xlsxwriter
import pandas as pd
import os
from mpire import WorkerPool
from google.cloud import bigquery
import csv
from collections import defaultdict
import matplotlib.pyplot as plt
import seaborn as sns 
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime


In [ ]:
cdr = os.environ["WORKSPACE_CDR"]
print(f"Workspace CDR: {cdr}")


In [ ]:
import pandas
import os

def get_pids_F01_F99(output_f):
    pid_sql = """
        SELECT
            person.person_id,
            person.gender_concept_id,
            person.sex_at_birth_source_value,
            person.gender_source_value,
            person.race_source_value,
            p_gender_concept.concept_name as gender,
            person.birth_datetime as date_of_birth,
            person.race_concept_id,
            p_race_concept.concept_name as race,
            person.ethnicity_concept_id,
            p_ethnicity_concept.concept_name as ethnicity,
            person.sex_at_birth_concept_id,
            p_sex_at_birth_concept.concept_name as sex_at_birth,
            person.self_reported_category_concept_id,
            p_self_reported_category_concept.concept_name as self_reported_category 
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.person` person 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_gender_concept 
                ON person.gender_concept_id = p_gender_concept.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_race_concept 
                ON person.race_concept_id = p_race_concept.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_ethnicity_concept 
                ON person.ethnicity_concept_id = p_ethnicity_concept.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_sex_at_birth_concept 
                ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_self_reported_category_concept 
                ON person.self_reported_category_concept_id = p_self_reported_category_concept.concept_id  
        WHERE
            person.PERSON_ID IN (SELECT
                distinct person_id  
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
            WHERE
                cb_search_person.person_id IN (SELECT
                    criteria.person_id 
                FROM
                    (SELECT
                        DISTINCT person_id, entry_date, concept_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                    WHERE
                        (concept_id IN(SELECT
                            DISTINCT c.concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                        JOIN
                            (SELECT
                                CAST(cr.id as string) AS id       
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                            WHERE
                                concept_id IN (1568106, 1568152, 1568206, 1568231, 1568264, 35207275, 1568088, 35207260, 1568181, 1568087, 35207116, 1568268, 1568221, 1568257, 1568248, 1568128, 35207292, 1568230, 35207210, 35207114, 1568259, 35207237, 35207253, 1568095, 35207243, 1568177, 35207242, 35207273, 1568141, 1568254, 1568164, 35207133, 1568262, 1568218, 35207276, 35207261, 1568239, 35207239, 1595539, 35207125, 35207134, 1568194, 35207132, 35207238, 1568249, 35207209, 35207174, 1568267, 35207166, 35207141, 1568266, 1568234, 1568265, 1568245, 1568093, 1568236, 1568209, 1568117, 35207140, 1568208, 1568260, 1568091, 1568242, 1568217, 35207240, 1568240, 35207115, 1568211, 1568220, 1568251, 35207208, 35207235, 35207241, 1568263, 35207135)       
                                AND full_text LIKE '%_rank1]%'      ) a 
                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                OR c.path LIKE CONCAT('%.', a.id) 
                                OR c.path LIKE CONCAT(a.id, '.%') 
                                OR c.path = a.id) 
                        WHERE
                            is_standard = 0 
                            AND is_selectable = 1) 
                        AND is_standard = 0 )) criteria ) )"""

    F01_F99_pid_df = pandas.read_gbq(
        pid_sql,
        dialect="standard",
        use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
        progress_bar_type="tqdm_notebook")
    F01_F99_pid_df.to_csv(output_f, index=False)
    return F01_F99_pid_df

In [ ]:
def get_all_participants(output_f, save_data=True):
    client = bigquery.Client() 
    query_sql = f"""
        SELECT
            person.person_id,
            -- person.gender_concept_id,
            p_gender_concept.concept_name AS gender,
            person.year_of_birth,
            -- person.race_concept_id,
            p_race_concept.concept_name AS race,
            -- person.ethnicity_concept_id,
            p_ethnicity_concept.concept_name AS ethnicity,
            -- person.sex_at_birth_concept_id,
            p_sex_at_birth_concept.concept_name AS sex_at_birth,
            obs_edu.value_source_value AS education_level,
            obs_marital.value_source_value AS marital_status,
            obs_insurance.value_source_value AS health_insurance,
            -- obs_employment.value_source_value AS employment_status, # do not include this as conflicting answers exist.
            obs_income.value_source_value AS income_level,
            obs_home.value_source_value AS current_home_status,
            obs_country.value_source_value AS country_origin,
            obs_house_concern.value_source_value AS house_concern,
            obs_living_years.value_source_value AS living_years
        FROM
            `{os.environ["WORKSPACE_CDR"]}.person` AS person
        LEFT JOIN
            `{os.environ["WORKSPACE_CDR"]}.concept` AS p_gender_concept
                ON person.gender_concept_id = p_gender_concept.concept_id
        LEFT JOIN
            `{os.environ["WORKSPACE_CDR"]}.concept` AS p_race_concept
                ON person.race_concept_id = p_race_concept.concept_id
        LEFT JOIN
            `{os.environ["WORKSPACE_CDR"]}.concept` AS p_ethnicity_concept
                ON person.ethnicity_concept_id = p_ethnicity_concept.concept_id
        LEFT JOIN
            `{os.environ["WORKSPACE_CDR"]}.concept` AS p_sex_at_birth_concept
                ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id
        LEFT JOIN
            `{os.environ["WORKSPACE_CDR"]}.observation` AS obs_edu
                ON person.person_id = obs_edu.person_id
                AND obs_edu.observation_source_concept_id = 1585940 -- 'Education Level: Highest Grade' 
        LEFT JOIN
            `{os.environ["WORKSPACE_CDR"]}.observation` AS obs_marital
                ON person.person_id = obs_marital.person_id
                AND obs_marital.observation_source_concept_id = 1585892 -- 'Marital status'
        LEFT JOIN
            `{os.environ["WORKSPACE_CDR"]}.observation` AS obs_insurance
                ON person.person_id = obs_insurance.person_id
                AND obs_insurance.observation_source_concept_id = 1585386 -- 'Health insurance'
        LEFT JOIN
            `{os.environ["WORKSPACE_CDR"]}.observation` AS obs_income
                ON person.person_id = obs_income.person_id
                AND obs_income.observation_source_concept_id = 1585375 -- 'Income level'
        LEFT JOIN
            `{os.environ["WORKSPACE_CDR"]}.observation` AS obs_home
                ON person.person_id = obs_home.person_id
                AND obs_home.observation_source_concept_id = 1585370 -- 'Current home own or rent'
        LEFT JOIN
            `{os.environ["WORKSPACE_CDR"]}.observation` AS obs_country
                ON person.person_id = obs_country.person_id
                AND obs_country.observation_source_concept_id = 1586135 -- 'Country origin/born'         
        LEFT JOIN
            `{os.environ["WORKSPACE_CDR"]}.observation` AS obs_house_concern
                ON person.person_id = obs_house_concern.person_id
                AND obs_house_concern.observation_source_concept_id = 1585886 -- 'Stable House Concern'
        LEFT JOIN
            `{os.environ["WORKSPACE_CDR"]}.observation` AS obs_living_years
                ON person.person_id = obs_living_years.person_id
                AND obs_living_years.observation_source_concept_id = 1585879 -- 'How Many Living Years in current place'  
    """

    query_job = client.query(query_sql)
    print("SQL query done!")    
    rows = query_job.result()

    columns = ['person_id', 'gender', 'year_of_birth', 'race', 'ethnicity', 'sex_at_birth', 
               'education_level', 'marital_status', 'health_insurance', 'income_level', 
               'current_home_status', 'country_origin',  'house_concern', 'living_years'] 

    
    demographics_df = pd.DataFrame(
        [(row.person_id, row.gender, row.year_of_birth, row.race, row.ethnicity, row.sex_at_birth, row.education_level, 
          row.marital_status, row.health_insurance, row.income_level, row.current_home_status, 
          row.country_origin, row.house_concern, row.living_years) 
         for row in rows],
        columns=columns
    )    

    if save_data:
        demographics_df.to_csv(output_f, index=False)
    
    print(f"Dataframe shape of all participants: {demographics_df.shape}.")
    return demographics_df

In [ ]:
sex_mapping = {
    'Female': 'Female',
    'Male': 'Male',
    'Intersex': 'Other',
    'Sex At Birth: Sex At Birth None Of These': 'Other',
    'No matching concept': 'Other',
    'I prefer not to answer': 'Other',
    #"Not male, not female, prefer not to answer, or skipped": "Unknown",
    'Other': 'Other'
    #None: 'Other'
}

gender_mapping = {
    'Female': 'Female',
    'Male': 'Male',
    'No matching concept': 'No matching concept',
    'Gender Identity: Transgender': 'Transgender',
    'Gender Identity: Non Binary': 'Non Binary',
    'Not man only, not woman only, prefer not to answer, or skipped': 'Other',
    'I prefer not to answer': 'Other',
    'Gender Identity: Additional Options': 'Other',
    'PMI: Skip': 'Other'
}

race_mapping = {
    'White': 'White',
    'Black or African American': 'Black or African American',
    'Asian': 'Another single population',
    'Native Hawaiian or Other Pacific Islander': 'Another single population',
    'American Indian or Alaska Native': 'Another single population',
    'More than one population': 'More than one population',
    'Middle Eastern or North African': 'Another single population',
    'None of these': 'Another single population',
    'I prefer not to answer': 'Other',
    'None Indicated': 'Other',
    'PMI: Skip': 'Other'  
}
ethnicity_mapping = {
                'Hispanic or Latino': 'Hispanic or Latino', 
                'What Race Ethnicity: Race Ethnicity None Of These': 'What Race Ethnicity: Race Ethnicity None Of These', 
                'Not Hispanic or Latino': 'Not Hispanic or Latino', 
                'No matching concept': 'No matching concept', 
                'PMI: Skip': 'Other',
                'PMI: Prefer Not To Answer': 'Other' }

HighestGrade_mapping = {'HighestGrade_CollegeGraduate': "College graduate", 
                        'HighestGrade_CollegeOnetoThree': "College one to three", 
                        'HighestGrade_NineThroughEleven': "Nine through eleven", 
                        'HighestGrade_TwelveOrGED': "Twelve or GED", 
                        'HighestGrade_AdvancedDegree': "Advanced degree", 
                        'HighestGrade_OneThroughFour': "One to eight or Never attended",
                        'HighestGrade_FiveThroughEight': "One to eight or Never attended",
                        'HighestGrade_NeverAttended': "One to eight or Never attended", 
                        'PMI_Skip': "Other", 
                        'PMI_PreferNotToAnswer': "Other", 
                        None: "Other"}

CurrentMaritalStatus_mapping = {'CurrentMaritalStatus_LivingWithPartner': "Living with partner", 
                                'CurrentMaritalStatus_Separated': "Separated", 
                                'CurrentMaritalStatus_Married': "Married", 
                                'CurrentMaritalStatus_NeverMarried': "Never married", 
                                'CurrentMaritalStatus_Widowed': "Widowed",
                                'CurrentMaritalStatus_Divorced': "Divorced",
                                'PMI_Skip': "Other",
                                'PMI_PreferNotToAnswer': "Other",
                                None: "Other"}


AnnualIncome_mapping = {'PMI_PreferNotToAnswer': "Other", 
                         'AnnualIncome_more200k': ">200k", 
                         'AnnualIncome_150k200k': "150k-200k", 
                         None: "Other", 
                         'AnnualIncome_35k50k': "35k-50k", 
                         'AnnualIncome_10k25k': "10k-25k", 
                         'PMI_Skip': "Other", 
                         'AnnualIncome_less10k': "<10k", 
                         'AnnualIncome_100k150k': "100k-150k", 
                         'AnnualIncome_25k35k': "25k-35k", 
                         'AnnualIncome_75k100k': "75k-100k", 
                         'AnnualIncome_50k75k': "50k-75k"}

name2dict = {'education_level': HighestGrade_mapping,
            'marital_status': CurrentMaritalStatus_mapping, 
             'income_level': AnnualIncome_mapping, 
             'gender': gender_mapping,
             'race': race_mapping,
             'ethnicity': ethnicity_mapping,
             'sex_at_birth': sex_mapping
}

In [ ]:
def age_group(age):
    if age < 18:
        return '<18'
    if age >= 18 and age <= 29:
        return '18-29'    
    elif (age >= 30 and age < 40):
        return '30-39'
    elif (age >= 40 and age < 50):
        return '40-49'
    elif (age >= 50 and age < 60):
        return '50-59'    
    elif (age >= 60 and age < 70):
        return '60-69'
    else:
        return '>=70'

In [ ]:
def get_mortality_from_death(output_f):
    """
    Fetches all unique cause_concept_id and cause_source_value from the 'death' table.

    Returns:
        pd.DataFrame: DataFrame containing all cause_concept_id and cause_source_value from death.
    """
    client = bigquery.Client()

    query = """
        SELECT
            d.person_id,
            d.death_date,
            d.death_type_concept_id,
            d.cause_concept_id,
            d.cause_source_value,
        FROM 
            `""" + os.environ["WORKSPACE_CDR"] + """.death` d          
        ORDER BY 
            d.person_id;
    """
    
    df = pd.read_gbq(query, dialect="standard")

    if df.empty:
        print("No death_type_concept_id found in death.")
        return pd.DataFrame()
    else:
        df.to_csv(output_f, index=False)
        print(f"Data saved to {output_f}.")

    return df

In [ ]:
def calculate_age(year_of_birth, death_date):
    cut_off_year = 2023

    death_year = death_date.year if pd.notna(death_date) else cut_off_year
    
    age = death_year - year_of_birth
    return age

In [ ]:
def get_drug_concept_ids(output_f):
    """
    Fetches all drug concept IDs from the 'concept' table based on the
    drug_type_concept_id values corresponding to prescription drugs.

    Args:
        output_f (str): File path to save the resulting CSV.

    Returns:
        pd.DataFrame: DataFrame containing drug concept IDs and their associated names.
    """
    client = bigquery.Client()

    query = """
        SELECT 
            c.concept_id,
            c.concept_name,
            atc.concept_id,
            atc.concept_code,
            atc.concept_name,
            atc.concept_class_id
        FROM 
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` c

        JOIN 
            `""" + os.environ["WORKSPACE_CDR"] + """.concept_relationship` cr
        ON 
            c.concept_id = cr.concept_id_1
        JOIN 
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` atc
        ON 
            cr.concept_id_2 = atc.concept_id            
        WHERE 
            c.vocabulary_id IN ('RxNorm', 'RxNorm Extension')
            AND c.valid_end_date > CURRENT_DATE()
            AND c.concept_class_id IN ('Ingredient')
            AND atc.vocabulary_id = 'ATC'
            AND atc.concept_code LIKE 'N%'
            AND atc.valid_end_date > CURRENT_DATE()
        GROUP BY 
            c.concept_id, c.concept_name, atc.concept_id, atc.concept_code, atc.concept_name, atc.concept_class_id
        ORDER BY 
            c.concept_name;
    """

    job_config = bigquery.QueryJobConfig()
    query_job = client.query(query, job_config=job_config)
    df = query_job.to_dataframe()

    if df.empty:
        print("No prescription drug concept IDs found.")
        return pd.DataFrame()
    else:
        print(f"The number of unique prescription drug concepts LIKE 'N%': {len(df)}")
        df.to_csv(output_f, index=False)
    return df

In [ ]:
def drug_exposure_atc(atc5_lst, output_f):   
    drug_sql = f"""
        SELECT 
            de.person_id,
            de.drug_exposure_id,
            de.drug_concept_id,
            de.drug_exposure_start_date,
            de.drug_exposure_end_date,
            -- de.verbatim_end_date, # The known end date of a Drug Exposure as provided by the source.
            de.drug_type_concept_id,
            -- de.stop_reason,
            -- de.refills,
            -- de.quantity,
            -- de.days_supply,
            -- de.sig,
            -- de.route_concept_id,
            -- de.lot_number,
            -- de.provider_id,
            -- de.visit_occurrence_id,
            -- de.visit_detail_id,
            -- de.drug_source_value,
            -- de.drug_source_concept_id,
            -- de.route_source_value,
            -- de.dose_unit_source_value,
            c.concept_name,
            c.concept_class_id,
            atc.concept_code as atc_code,
            atc.concept_name as atc_name
        FROM 
            `""" + os.environ["WORKSPACE_CDR"] + """.drug_exposure` de
        JOIN 
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` c 
        ON 
            de.drug_concept_id = c.concept_id
        JOIN 
            `""" + os.environ["WORKSPACE_CDR"] + """.concept_relationship` cr
        ON 
            de.drug_concept_id = cr.concept_id_1
        JOIN 
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` atc
        ON 
            cr.concept_id_2 = atc.concept_id 
            AND atc.vocabulary_id = 'ATC'
            AND atc.concept_code LIKE 'N%'
            AND atc.valid_end_date > CURRENT_DATE()            
        WHERE
            c.vocabulary_id IN ('RxNorm', 'RxNorm Extension')
            AND c.valid_end_date > CURRENT_DATE()
            AND c.concept_class_id IN ('Ingredient')
            AND atc.concept_code IN UNNEST(@atc_lst)
            -- AND de.drug_type_concept_id IN (32838, 32839, 32825, 32837)581373,  32869, 32833, 
            -- AND de.drug_type_concept_id IN (32825, 32830, 32837, 32838, 32839, 32865,581452, 38000175, 38000176, 38000177, 38000178, 38000180, 44787730)
            AND cr.relationship_id IN ('RxNorm - ATC pr lat')
        ORDER BY 
            de.person_id, de.drug_exposure_start_date
        """
    client = bigquery.Client()
    
    atc_lst = [str(atc) for atc in atc5_lst]
    
    job_config = bigquery.QueryJobConfig(
        query_parameters=[bigquery.ArrayQueryParameter("atc_lst", "STRING", atc_lst)]
    ) 
    query_job = client.query(drug_sql, job_config=job_config)
    print("SQL query done!")    
    rows = query_job.result()        
    
    columns = ['person_id', 'drug_exposure_id', 'drug_concept_id', 'drug_exposure_start_date', 'drug_exposure_end_date',
               'drug_type_concept_id', 'concept_name', 'concept_class_id', 
               "atc_code", "atc_name"]
    
    drug_df = pd.DataFrame(
        [(row.person_id, row.drug_exposure_id, row.drug_concept_id, row.drug_exposure_start_date, 
          row.drug_exposure_end_date, row.drug_type_concept_id, row.concept_name, row.concept_class_id, 
          row.atc_code, row.atc_name ) 
         for row in rows],
        columns=columns
    ) 
       
    drug_df.to_parquet(output_f)
    return drug_df

In [ ]:
def get_all_atc_N_2name(output_f):
    """
    Fetches all unique atc code and names from the 'concept' table.

    Returns:
        pd.DataFrame: DataFrame containing all unique code and names from concept.
    """
    client = bigquery.Client()

    query = """
        SELECT
            c.concept_id,
            c.concept_code,
            c.concept_name, 
            c.concept_class_id
        FROM 
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` c
        WHERE
            c.vocabulary_id = 'ATC'
            AND c.concept_code LIKE 'N%'
            AND c.valid_end_date > CURRENT_DATE()            
        ORDER BY 
            c.concept_code;
    """
    
    df = pd.read_gbq(query, dialect="standard")

    if df.empty:
        print("No drug_type_concept_ids found in drug_exposure.")
        return pd.DataFrame()
    else:
        df.to_csv(output_f, index=False)
        print(f"Data saved to {output_f}.")

    return df

In [ ]:
def atc2name(atc_df, atc_code):
    filtered_df = atc_df.loc[atc_df["concept_code"] == atc_code, "concept_name"]
    if filtered_df.empty:
        return "Unknown" 
    else:
        return filtered_df.iloc[0]

In [ ]:
import statsmodels.stats.proportion as proportion

def calculate_percentage_ci(num, sample_size, confidence_level=0.95):
    print(num, sample_size)
    p = num/sample_size

    (lower, upper) = proportion.proportion_confint(num, sample_size, alpha=1-confidence_level, method='wilson')

    percentage = "{:.1f}".format(p * 100)
    lower_bound = "{:.1f}".format(lower * 100)  
    upper_bound = "{:.1f}".format(upper * 100)      
    return percentage,lower_bound, upper_bound

In [ ]:
from datetime import timedelta

def visit_count(df, follow_up_days):
    if not df.empty:
        df.loc[:, 'visit_start_date'] = pd.to_datetime(df['visit_start_date'])
        first_date = df['visit_start_date'].min() 
        if follow_up_days == None:
            visit_count = df['visit_start_date'].nunique()
            return visit_count, first_date
        else:
            cutoff_date = first_date + timedelta(days=follow_up_days)    
            visit_count_before_cutoff = df[df['visit_start_date'] < cutoff_date]['visit_start_date'].nunique()
            return visit_count_before_cutoff, first_date
    else:
        return 0, None

In [ ]:
def person_visit(person_visit_df, follow_up_days=None):
    
    ERV_df = person_visit_df[person_visit_df['concept_name'] == "Emergency Room Visit"]
    IV_df = person_visit_df[person_visit_df['concept_name'] == "Inpatient Visit"]
    
    ERV_count, ERV_first_date = visit_count(ERV_df, follow_up_days)
    IV_count, IV_first_date = visit_count(IV_df, follow_up_days)
    
    return (ERV_count, ERV_first_date, IV_count, IV_first_date)

def get_visit_count(merged_df, n_jobs, output_f, save_data=True):
    person_ids = list(set(merged_df['person_id'].to_list()))
    def hospitalization_visit(merged_df, person_id):
        person_visit_df = merged_df[merged_df['person_id'] == person_id]
        return person_visit(person_visit_df)  # Process the person's data   
    
    with WorkerPool(n_jobs=n_jobs, shared_objects=merged_df) as pool:
        results = pool.map(hospitalization_visit, person_ids)
        
    columns = ['person_id', 'ERV_count', 'ERV_first_date', "IV_count", 'IV_first_date']
    visit_count_df = pd.DataFrame(columns=columns)
    
    rows = []
    for i in range(len(person_ids)):
        p_id = person_ids[i]
        (ERV_count, ERV_first_date, IV_count, IV_first_date) = results[i]     
        row = [p_id, ERV_count, ERV_first_date, IV_count, IV_first_date]
        
        rows.append([p_id, ERV_count, ERV_first_date, IV_count, IV_first_date])
    visit_count_df = pd.DataFrame(rows, columns=columns)
    
    if save_data and not visit_count_df.empty:
        visit_count_df.to_csv(output_f, index=False)
    return visit_count_df

In [ ]:
import ast
import ast

def count_concurrent_medications(df, window_size):
    """
    Count concurrent unique drug groups at each unique start date.
    
    Parameters:
        df (pd.DataFrame): Dataframe with ['drug_exposure_start_date', 'drug_exposure_end_date', 'atc_code']
        window_size (pd.Timedelta): Time window for counting concurrent use
    
    Returns:
        pd.DataFrame: Dataframe with columns ['drug_exposure_start_date', 'concurrent_count', 'merged_drug_groups']
    """
    results = []
    
    df = df.copy()
    df["drug_exposure_start_date"] = pd.to_datetime(df["drug_exposure_start_date"])
    df["drug_exposure_end_date"] = pd.to_datetime(df["drug_exposure_end_date"])

    unique_start_dates = sorted(df["drug_exposure_start_date"].unique())
    
    for start in unique_start_dates:
        
        new_df = df[(df["drug_exposure_start_date"] <= start) & (df["drug_exposure_end_date"] >= start + pd.Timedelta(days=window_size))]
        
        active_set_lst = new_df["atc_code"].to_list()

        concurrent_count = len(set(active_set_lst))
        # Store results
        results.append({
            "drug_exposure_start_date": start,
            "concurrent_count": concurrent_count,
            "merged_drug_groups": sorted(set(active_set_lst))
        })

    return pd.DataFrame(results)

def safe_eval(val):
    if isinstance(val, str):
        return ast.literal_eval(val)
    return val

def get_max_concurrent_medications(result_df):
    if result_df.empty:
        return (0, [])

    result_df = result_df.copy()

    result_df["drug_exposure_start_date"] = pd.to_datetime(
        result_df["drug_exposure_start_date"]
    )

    result_df["merged_drug_groups"] = result_df["merged_drug_groups"].apply(safe_eval)

    max_count = result_df["concurrent_count"].max()

    max_rows = result_df[result_df["concurrent_count"] == max_count]
    
    unique_drug_groups = set()
                
    for groups in max_rows["merged_drug_groups"]:
        unique_drug_groups.update(groups)

    return (max_count, sorted(unique_drug_groups))

In [ ]:
import ast

def count_concurrent_medications(df, window_size):
    """
    Count concurrent unique drug groups at each unique start date.
    
    Parameters:
        df (pd.DataFrame): Dataframe with ['drug_exposure_start_date', 'drug_exposure_end_date', 'atc_code']
        window_size (pd.Timedelta): Time window for counting concurrent use
    
    Returns:
        pd.DataFrame: Dataframe with columns ['drug_exposure_start_date', 'concurrent_count', 'merged_drug_groups']
    """
    results = []
    
    df = df.copy()
    df["drug_exposure_start_date"] = pd.to_datetime(df["drug_exposure_start_date"])
    df["drug_exposure_end_date"] = pd.to_datetime(df["drug_exposure_end_date"])

    unique_start_dates = sorted(df["drug_exposure_start_date"].unique())
    
    for start in unique_start_dates:
        
        new_df = df[(df["drug_exposure_start_date"] <= start) & (df["drug_exposure_end_date"] >= start + pd.Timedelta(days=window_size))]
        
        active_set_lst = new_df["atc_code"].to_list()

        concurrent_count = len(set(active_set_lst))
        # Store results
        results.append({
            "drug_exposure_start_date": start,
            "concurrent_count": concurrent_count,
            "merged_drug_groups": sorted(set(active_set_lst))
        })

    return pd.DataFrame(results)

def safe_eval(val):
    if isinstance(val, str):
        return ast.literal_eval(val)
    return val

def get_max_concurrent_medications(result_df):
    if result_df.empty:
        return (0, [])

    result_df = result_df.copy()

    result_df["drug_exposure_start_date"] = pd.to_datetime(
        result_df["drug_exposure_start_date"]
    )

    result_df["merged_drug_groups"] = result_df["merged_drug_groups"].apply(safe_eval)

    # max concurrency
    max_count = result_df["concurrent_count"].max()

    max_rows = result_df[result_df["concurrent_count"] == max_count]
    
    unique_drug_groups = set()
                
    for groups in max_rows["merged_drug_groups"]:
        unique_drug_groups.update(groups)

    return (max_count, sorted(unique_drug_groups))

In [ ]:
def get_polypharmacy(person_df, window_size):
    total_set_lst = person_df["atc_code"].to_list()

    total_set = set(total_set_lst)
    life_time_polypharmacy = len(total_set)
    
    first_date = person_df['drug_exposure_start_date'].min() 
    last_date = person_df['drug_exposure_end_date'].max() 
    duration = (last_date - first_date).days
    
    concurrent_polypharmacy_df = count_concurrent_medications(person_df, window_size)

    (max_polypharmacy, max_polypharmacy_atc_lst) = get_max_concurrent_medications(concurrent_polypharmacy_df)

    return (life_time_polypharmacy, total_set, duration, first_date, last_date, max_polypharmacy, max_polypharmacy_atc_lst)

def polypharmacy_calculation(merged_df, n_jobs, output_file, window_size=30):
    person_ids = list(set(merged_df['person_id'].to_list()))    
    def get_polypharmacy_from_df(shared_objects, person_id):
        merged_df, window_size = shared_objects
        person_df = merged_df[merged_df['person_id'] == person_id]
        
        return get_polypharmacy(person_df, window_size)  # Process the person's data 
        
    shared_objs = merged_df, window_size
    with WorkerPool(n_jobs=n_jobs, shared_objects=shared_objs) as pool:
        results = pool.map(get_polypharmacy_from_df, person_ids)
        
    columns = ['person_id', 'lifetime_polypharmacy', "total_atc_codes", "duration", 'first_drug_exposure', 'last_drug_exposure', 'concurrent_polypharmacy', 'concurrent_polypharmacy_atc_codes']#, "medication_switch_lst"]
    polypharmacy_df = pd.DataFrame(columns=columns)
     
    unit_polypharmacy_columns = set()
    
    polypharmacy_dict = {}
    for i in range(len(person_ids)):
        p_id = person_ids[i]
        (life_time_polypharmacy, total_atc, duration, first_date, last_date, max_polypharmacy, max_polypharmacy_atc) = results[i] 
            
        row = [p_id, life_time_polypharmacy, total_atc, duration, first_date, last_date, max_polypharmacy, max_polypharmacy_atc]
        # print(row)
        polypharmacy_df.loc[len(polypharmacy_df)] = row
            
    if not polypharmacy_df.empty:
        polypharmacy_df.to_csv(output_file, index=False)
    return polypharmacy_df


In [ ]:
# def get_polypharmacy(person_df, window_size):
#     total_set_lst = person_df["atc_code"].to_list()

#     total_set = set(total_set_lst)
#     life_time_polypharmacy = len(total_set)
    
#     first_date = person_df['drug_exposure_start_date'].min() 
#     last_date = person_df['drug_exposure_end_date'].max() 
#     duration = (last_date - first_date).days
    
#     concurrent_polypharmacy_df = count_concurrent_medications(person_df, window_size)

#     (max_polypharmacy, max_polypharmacy_atc_lst) = get_max_concurrent_medications(concurrent_polypharmacy_df)

#     return (life_time_polypharmacy, total_set, duration, first_date, last_date, max_polypharmacy, max_polypharmacy_atc_lst)

# def polypharmacy_calculation(merged_df, n_jobs, output_file, window_size=30):
#     person_ids = list(set(merged_df['person_id'].to_list()))    
#     def get_polypharmacy_from_df(shared_objects, person_id):
#         merged_df, window_size = shared_objects
#         person_df = merged_df[merged_df['person_id'] == person_id]
        
#         return get_polypharmacy(person_df, window_size)  # Process the person's data 
        
#     shared_objs = merged_df, window_size
#     with WorkerPool(n_jobs=n_jobs, shared_objects=shared_objs) as pool:
#         results = pool.map(get_polypharmacy_from_df, person_ids)
        
#     columns = ['person_id', 'lifetime_polypharmacy', "total_atc_codes", "duration", 'first_drug_exposure', 'last_drug_exposure', 'concurrent_polypharmacy', 'concurrent_polypharmacy_atc_codes']#, "medication_switch_lst"]
#     polypharmacy_df = pd.DataFrame(columns=columns)
     
#     unit_polypharmacy_columns = set()
    
#     polypharmacy_dict = {}
#     for i in range(len(person_ids)):
#         p_id = person_ids[i]
#         (life_time_polypharmacy, total_atc, duration, first_date, last_date, max_polypharmacy, max_polypharmacy_atc) = results[i]
            
#         row = [p_id, life_time_polypharmacy, total_atc, duration, first_date, last_date, max_polypharmacy, max_polypharmacy_atc]
#         polypharmacy_df.loc[len(polypharmacy_df)] = row
            
#     if not polypharmacy_df.empty:
#         polypharmacy_df.to_csv(output_file, index=False)
#     return polypharmacy_df


In [ ]:
def get_mental_disorder_icd(output_f):
    client = bigquery.Client()

    query = """
        SELECT
            c.concept_id,
            c.concept_code,
            c.concept_name 
        FROM 
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` c
        WHERE
            c.vocabulary_id IN ('ICD9CM','ICD10CM')
            AND c.domain_id = 'Condition'
            AND c.valid_end_date > CURRENT_DATE()
            AND c.concept_code LIKE 'F%'
        ORDER BY 
            c.concept_code;
    """  
    df = pd.read_gbq(query, dialect="standard")

    if df.empty:
        print("No ICD10CM codes found.")
        return pd.DataFrame()
    else:
        df.to_csv(output_f, index=False)
        print(f"Data saved to {output_f}.")

    return df

In [ ]:
def extract_person_ids_omops(concept_ids, output_f, save_data=True):
    client = bigquery.Client()
    concept_ids = [int(cid) for cid in concept_ids]

    person_sql = """
        SELECT
            person.person_id
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.person` person   
        WHERE
            person.PERSON_ID IN (SELECT
                distinct person_id  
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
            WHERE
                cb_search_person.person_id IN (SELECT
                    criteria.person_id 
                FROM
                    (SELECT
                        DISTINCT person_id, entry_date, concept_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                    WHERE
                        (concept_id IN(SELECT
                            DISTINCT c.concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                        JOIN
                            (SELECT
                                CAST(cr.id as string) AS id       
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                            WHERE
                                concept_id IN UNNEST(@concept_ids)      
                                AND full_text LIKE '%_rank1]%'      ) a 
                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                OR c.path LIKE CONCAT('%.', a.id) 
                                OR c.path LIKE CONCAT(a.id, '.%') 
                                OR c.path = a.id) 
                        WHERE
                            is_standard = 0 
                            AND is_selectable = 1) 
                        AND is_standard = 0 )) criteria ) )"""
    job_config = bigquery.QueryJobConfig(
        query_parameters=[bigquery.ArrayQueryParameter("concept_ids", "INT64", concept_ids)]
    ) 

    query_job = client.query(person_sql, job_config=job_config)
    print("SQL query done!")    
    rows = query_job.result()
    
    columns = ['person_id']

    icd_person_df = pd.DataFrame(
        [(row.person_id) for row in rows],
        columns=columns
    )   
    if save_data:
        icd_person_df.to_csv(output_f, index=False)
    print(f"Dataframe shape: {icd_person_df.shape}.")
    return icd_person_df

In [ ]:
def charlson_index(person_ids, icd_code_concept_df, output_dir):
    group2icdcodes = {
        "Myocardial_infarction": ["I21", "I22", "I25.2"],
        "Congestive_heart_failure": ["I11.0", "I13.0", "I13.2", "I25.5", "I42.0", "I42.6", "I42.7", "I42.8", "I42.9", "I4.3", "I50"],
        "Peripheral_vascular_disease": ["I70", "I71", "I73.1", "I73.8", "I73.9", "I77.1", "I79.0", "I79.2", "K55"],
        "Cerebrovascular_disease": ["G45", "I60", "I61", "I62", "I63", "I64", "I67", "I69"],
        "Chronic_obstructive_pulmonary_disease": ["J43", "J44"],
        "Chronic_other_pulmonary_disease": ["J41", "J42", "J45", "J46", "J47", "J60", "J61", "J62", "J63", "J64", "J65", "J66", "J67", "J68", "J69", "J70"],
        "Rheumatic_disease": ['M05', 'M06', 'M12.3', 'M07.0', 'M07.1', 'M07.2', 'M07.3', 'M08', 'M13', 'M30', 'M31.3', 'M31.4', 'M31.5', 'M31.6', 'M32', 'M33', 'M34', 'M35.0', 'M35.1', 'M35.3', 'M45', 'M46'],
        "Dementia": ["F00", "F01", "F02", "F03", "F05.1", "G30", "G31.1", "G31.9"],
        "Hemiplegia": ["G11.4", "G80", "G81", "G82", "G83.0", "G83.1", "G83.2", "G83.3", "G83.8"],
        "Diabetes_without_chronic_complication": ["E10.0", "E10.1", "E11.0", "E11.1", "E12.0", "E12.1", "E13.0", "E13.1", "E14.0", "E14.1"],
        "Diabetes_with_chronic_complication": ["E10.2", "E10.3", "E10.4", "E10.5", "E10.7", "E11.2", "E11.3", "E11.4", "E11.5", "E11.6", "E11.7", "E12.2", "E12.3", "E12.4", "E12.5", "E12.6", "E12.7", "E13.2", "E13.3", "E13.4", "E13.5", "E13.6", "E13.7", "E14.2", "E14.3", "E14.4", "E14.5", "E14.6", "E14.7"],
        "Renal_disease": ["I12.0", "I13.1", "N03.2", "N03.3", "N03.4", "N03.5", "N03.6", "N03.7", "N05.2", "N05.3", "N05.4", "N05.5", "N05.6", "N05.7", "N11", "N18", "N19", "N25.0", "Q61.1", "Q61.2", "Q61.3", "Q61.4", "Z49", "Z94.0", "Z99.2"],
        "Mild_liver_disease": ["B15", "B16", "B17", "B18", "B19", "K70.3", "K70.9", "K73", "K74.6", "K75.4"],
        "liver_special": ["R18"],
        "moderate_severe_liver_disease": ["I85.0", "I85.9", "I98.2", "I98.3"],
        "Peptic_ulcer_disease": ["K25", "K26", "K27", "K28"],
        "Malignancy": ["C00", "C01", "C02", "C03", "C04", "C05", "C06", "C07", "C08", "C09", "C10", "C11", "C12", "C13", "C14", "C15", "C16", "C17", "C18", "C19", "C20", "C21", "C22", "C23", "C24", "C25", "C26", "C27", "C28", "C29", "C30", "C31", "C32", "C33", "C34", "C35", "C36", "C37", "C38", "C39", "C40", "C41", "C43", "C45", "C46", "C47", "C48", "C49", "C50", "C51", "C52", "C53", "C54", "C55", "C56", "C57", "C58", "C60", "C61", "C62", "C63", "C64", "C65", "C66", "C67", "C68", "C69", "C70", "C71", "C72", "C73", "C74", "C75", "C76", "C81", "C82", "C83", "C84", "C85", "C86", "C88", "C89", "C90", "C91", "C92", "C93", "C94", "C95", "C96", "C97"],
        "Metastatic_cancer": ["C77", "C78", "C79", "C80"],
        "Aids": ["B20", "B21", "B22", "B23", "B24", "F02.4", "O98.7", "R75", "Z21.9", "Z71.7"]
    }

    weights = {
        "Myocardial_infarction": 1,
        "Congestive_heart_failure": 1,
        "Peripheral_vascular_disease": 1,
        "Cerebrovascular_disease": 1,
        "Chronic_obstructive_pulmonary_disease": 1,
        "Chronic_other_pulmonary_disease": 1,
        "Rheumatic_disease": 1,
        "Dementia": 1,
        "Hemiplegia": 2,
        "Diabetes_without_chronic_complication": 1,
        "Diabetes_with_chronic_complication": 2,
        "Renal_disease": 2,
        "Mild_liver_disease": 1,
        "liver_special": 1,
        "moderate_severe_liver_disease": 3,
        "Peptic_ulcer_disease": 1,
        "Malignancy": 2,
        "Metastatic_cancer": 6,
        "Aids": 6
    }    
    ###############################################################################
    # modify the charlson condition group dictionary as some of the ICD codes do not appear in this cohort
    cohort_icd_codes = icd_code_concept_df['concept_code'].to_list()
    modified_group2icdcodes = {}
    for condition, icd_codes in group2icdcodes.items():
        modified_group2icdcodes[condition] = list(set(icd_codes) & set(cohort_icd_codes))
    ##############################################################################
    charlson2pids = {}
    for condition_group, icd_codes in modified_group2icdcodes.items():
        print(icd_codes)
        cur_concept_ids = []
        for code in icd_codes:
            concept_id = icd_code_concept_df.loc[icd_code_concept_df["concept_code"] == code, "concept_id"].values[0]
            cur_concept_ids.append(concept_id)
        output_f = f"{output_dir}/data/charlson/icd10cm_{condition_group}_person_ids.csv"
        icd_person_df = extract_person_ids_omops(cur_concept_ids, output_f, save_data=True)        
        icd_person_df = pd.read_csv(output_f)
        cur_condition_pids = icd_person_df['person_id'].to_list()    
        charlson2pids[condition_group] = cur_condition_pids
    
    column_names = list(charlson2pids.keys())
    pid_charlson_df = pd.DataFrame(0, index=person_ids, columns=column_names)
    
    for condition, pids in charlson2pids.items():
        pid_charlson_df.loc[pid_charlson_df.index.isin(pids), condition] = 1
        
    pid_charlson_df.reset_index(inplace=True)
    pid_charlson_df.rename(columns={'index': 'person_id'}, inplace=True)
        
    pid_charlson_df['Multimorbidity'] = pid_charlson_df[column_names].sum(axis=1)
    pid_charlson_df['CCI'] = pid_charlson_df.apply(lambda row: sum(row[col] * weights[col] for col in weights), axis=1)

    return pid_charlson_df[['person_id', 'Multimorbidity', 'CCI']]

In [ ]:
def get_condition_icd_3_4_char_code_name(output_f):
    client = bigquery.Client()

    query = """
        SELECT
            c.concept_id,
            c.concept_code,
            c.concept_name 
        FROM 
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` c
        WHERE
            c.vocabulary_id IN ('ICD9CM','ICD10CM')
            AND c.concept_class_id IN ('3-char nonbill code', '4-char nonbill code')
            AND c.domain_id = 'Condition'
            AND c.valid_end_date > CURRENT_DATE()            
        ORDER BY 
            c.concept_code;
    """  
    df = pd.read_gbq(query, dialect="standard")

    if df.empty:
        print("No ICD9CM/ICD10CM codes found.")
        return pd.DataFrame()
    else:
        df.to_csv(output_f, index=False)
        print(f"Data saved to {output_f}.")

    return df

In [ ]:
def add_binary(df, column_s, column_t, threshold):
    df[column_t] = np.where(df[column_s].isna(), np.nan, (df[column_s] >= threshold).astype(int))
    return df

In [ ]:
def get_hospitalisation(person_ids, output_f, save_data=True):
    client = bigquery.Client()
    person_ids = [int(pid) for pid in person_ids]    
    person_sql = """
        SELECT
            vc.person_id as person_id,
            c.concept_id as concept_id,
            c.concept_name as concept_name,
            vc.visit_start_date as visit_start_date,
            vc.visit_end_date as visit_end_date,
            -- vc.visit_source_value, -- this is code
            -- vc.visit_source_concept_id,
            -- vc.admitting_source_concept_id,
            -- vc.admitting_source_value,
            -- vc.discharge_to_concept_id,
            -- vc.discharge_to_source_value,
            -- vc.preceding_visit_occurrence_id
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.visit_occurrence` vc
        JOIN 
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` c
        ON 
            vc.visit_concept_id = c.concept_id 
        WHERE
            vc.person_id IN UNNEST(@person_ids) 
            AND c.vocabulary_id = "Visit"
            AND c.valid_end_date > CURRENT_DATE()
            AND c.concept_class_id = "Visit"
            AND c.concept_id IN (9201,9203) -- 32037, 38004515, 8717, 8870, 262
        ORDER BY
            vc.person_id, vc.visit_start_date
    """
    
    job_config = bigquery.QueryJobConfig(
        query_parameters=[bigquery.ArrayQueryParameter("person_ids", "INT64", person_ids)]
    ) 
    
    query_job = client.query(person_sql, job_config=job_config)
    print("SQL query done!")    
    rows = query_job.result()

    columns = ['person_id', 'concept_id', 'concept_name', 'visit_start_date', 'visit_end_date']

    visit_person_df = pd.DataFrame(
        [(row.person_id, row.concept_id, row.concept_name, row.visit_start_date, row.visit_end_date) for row in rows],
        columns=columns
    )         
    if save_data:
        visit_person_df.to_csv(output_f, index=False)
    return visit_person_df

In [ ]:
import pandas
import os

def get_short_read_WGS_participants(output_f, save_data):
    person_sql = """
        SELECT
            person.person_id,
            person.gender_concept_id,
            p_gender_concept.concept_name as gender,
            person.birth_datetime as date_of_birth,
            person.race_concept_id,
            p_race_concept.concept_name as race,
            person.ethnicity_concept_id,
            p_ethnicity_concept.concept_name as ethnicity,
            person.sex_at_birth_concept_id,
            p_sex_at_birth_concept.concept_name as sex_at_birth,
            person.self_reported_category_concept_id,
            p_self_reported_category_concept.concept_name as self_reported_category 
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.person` person 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_gender_concept 
                ON person.gender_concept_id = p_gender_concept.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_race_concept 
                ON person.race_concept_id = p_race_concept.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_ethnicity_concept 
                ON person.ethnicity_concept_id = p_ethnicity_concept.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_sex_at_birth_concept 
                ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id 
        LEFT JOIN
            `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_self_reported_category_concept 
                ON person.self_reported_category_concept_id = p_self_reported_category_concept.concept_id  
        WHERE
            person.PERSON_ID IN (SELECT
                distinct person_id  
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
            WHERE
                cb_search_person.person_id IN (SELECT
                    person_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
                WHERE
                    has_whole_genome_variant = 1 ) )"""

    person_df = pandas.read_gbq(
        person_sql,
        dialect="standard",
        use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
        progress_bar_type="tqdm_notebook")
    if save_data:
        person_df.to_csv(output_f, index=False)
    print(f"Dataframe shape: {person_df.shape}.")
    return person_df

In [ ]:
## Table 
from collections import Counter

def medication_distribution(atc_N_code2name_df, target_df, TOTAL_count, output_f, polypharmacy_threshold, target_column='lifetime_polypharmacy', top_n=None):
    if target_column == 'lifetime_polypharmacy':
        target_column2 = "total_atc_codes" 
        fig_prefix = "lifetime_polypharmacy_binary"
    else:
        target_column2 = "concurrent_polypharmacy_atc_codes"
        fig_prefix = "concurrent_polypharmacy_binary"

    selected_columns = ["person_id", target_column, target_column2]

    polypharmacy_df = target_df[target_df[target_column] >= polypharmacy_threshold][selected_columns].copy()
    #########################
    # print(polypharmacy_df[target_column2])
    # polypharmacy_df[target_column2] = polypharmacy_df[target_column2].apply(ast.literal_eval)
    list_of_sets = polypharmacy_df[target_column2].tolist()
    # print(list_of_sets)
    frozen_sets = [frozenset(ss) for ss in list_of_sets]

    # Count occurrences
    counts = Counter(frozen_sets)
    print(len(counts))
    # If you want output as normal sets with their counts:
#     for ss, count in counts.items():
#         print(set(ss), "->", count)
    ##########################
    total = len(polypharmacy_df)
    print(total)
    all_medication_lst = []
    for index, row in polypharmacy_df.iterrows():
#         parsed_list = ast.literal_eval(row[target_column2])
#         flattened_list = list(set([item for s in parsed_list for item in s]))
#         flattened_list = list(literal_eval(row[target_column2]))
        flattened_list = row[target_column2]
#         print(row[target_column2])
#         print(flattened_list)
        
        all_medication_lst.extend(flattened_list)
    

    counter = Counter(all_medication_lst)

    tops = counter.most_common(top_n)
    print(tops)
    result_df = pd.DataFrame(columns = ["Rank", "Medication (ATC 5th)", f"Participants, % (N={total})", "Class (ATC 3rd level)"])
    rank = 1
    for (atc_code, count) in tops:
        atc_name = atc2name(atc_N_code2name_df, atc_code)
        ptg_str = "{:.2f}".format(count/total * 100)
        
        atc_3 = atc_code[:4]
        atc_3_name = atc2name(atc_N_code2name_df, atc_3)
        
        row = [rank, f"{atc_name} ({atc_code})",ptg_str, f"{atc_3_name} ({atc_3})"]
        rank += 1
        result_df.loc[len(result_df)] = row
        
    result_df.to_excel(output_f)
    return result_df

In [ ]:
# polypharmacy by multimorbidity, co-diagnosis of mental health disorders
def generate_upset_data(polypharmacy_df, output_file, threshold=2, top_comb_n = 20):
    
    multi_diagnosis_df = polypharmacy_df[polypharmacy_df['psychiatric_diagnoses_count']>=threshold]    
    
    top_diagnosis_comb_list = multi_diagnosis_df['psychiatric_diagnoses'].value_counts().head(top_comb_n).index.tolist()    
    
    filtered_df = polypharmacy_df[polypharmacy_df['psychiatric_diagnoses'].isin(top_diagnosis_comb_list)]
    
    pids = filtered_df['person_id'].tolist()
    top_diagnosis_icds = set([item for sublist in top_diagnosis_comb_list for item in sublist])
    
    unique_icds = set()
    for s in top_diagnosis_comb_list:
        #icd = ast.literal_eval(s)
        unique_icds.update(s)
    print(unique_icds)
    
    unique_icd_lst = list(unique_icds)
    upset_df = pd.DataFrame(False, index=sorted(pids), columns=sorted(unique_icd_lst))
    
    #print(filtered_df['psychiatric_diagnoses'])
    
    for pid in pids:
        #print(pid)
        #icd_lst = ast.literal_eval(filtered_df[filtered_df['person_id']==pid]['psychiatric_diagnoses'].tolist()[0])
        icd_lst = filtered_df[filtered_df['person_id'] == pid]['psychiatric_diagnoses'].tolist()[0]

        for icd in icd_lst:         
            upset_df.loc[pid, icd] = True

    print(upset_df)
    print(len(upset_df))
    upset_df = upset_df.loc[~(upset_df.eq(False).all(axis=1))]

    upset_df = upset_df[upset_df.any(axis=1)]
    upset_df = upset_df.astype(int)    

    upset_df.to_csv(output_file)
    print(upset_df)
    return upset_df
